In [5]:
import Pkg; Pkg.add("ExponentialUtilities")

   Resolving package versions...
    Updating `/opt/julia/environments/v1.9/Project.toml`
⌅ [d4d017d3] + ExponentialUtilities v1.25.0
  No Changes to `/opt/julia/environments/v1.9/Manifest.toml`


In [9]:
using LinearAlgebra
using ExponentialUtilities  # 行列指数関数を計算

# スキュー対称行列を生成 (so(3)の元)
function hat(v::AbstractVector{<:Real})
    return [
        0.0    -v[3]   v[2];
        v[3]   0.0    -v[1];
       -v[2]   v[1]   0.0
    ]
end

# 交換子: [A, B] = AB - BA
function commutator(A::Matrix, B::Matrix)
    return A * B - B * A
end

# ad(X) を行列として表現 (so(3) 基底上の線形変換)
function ad_matrix(X::Matrix)
    # so(3)の標準基底
    E1 = hat([1.0, 0.0, 0.0])
    E2 = hat([0.0, 1.0, 0.0])
    E3 = hat([0.0, 0.0, 1.0])
    basis = [E1, E2, E3]

    # ad(X) の作用を基底上に展開し 3x3 行列を構成
    A = zeros(3, 3)
    for i in 1:3
        comm = commutator(X, basis[i])
        for j in 1:3
            A[j, i] = 0.5 * tr(comm' * basis[j])  # inner product: tr(AᵗB)
        end
    end
    return A
end

# メイン処理
v = [1.0, 2.0, 3.0]
X = hat(v)             # so(3) の元
t = 0.1

# 任意の Y ∈ so(3)
Y = hat([0.5, -1.0, 0.5])
Y_vec = [0.5, -1.0, 0.5]  # so(3) 基底の係数としての Y

# Ad(exp(tX))(Y)
g = exp(t * X)
Ad_exp_tX_Y = g * Y * g'

# exp(t * ad(X))(Y)
adX = ad_matrix(X)
exp_adX = exp(t * adX)
exp_adX_Y_vec = exp_adX * Y_vec

# ベクトルから so(3) 行列に戻す
E1, E2, E3 = hat.([[1.0, 0, 0], [0, 1.0, 0], [0, 0, 1.0]])
exp_adX_Y = exp_adX_Y_vec[1] * E1 + exp_adX_Y_vec[2] * E2 + exp_adX_Y_vec[3] * E3

# 結果出力
println("Ad(exp(tX))(Y):")
display(Ad_exp_tX_Y)

println("\nexp(t ad(X))(Y):")
display(exp_adX_Y)

println("\n一致するか？ -> ", isapprox(Ad_exp_tX_Y, exp_adX_Y; atol=1e-10))


Ad(exp(tX))(Y):


3×3 Matrix{Float64}:
 0.0       -0.270041     -0.83313
 0.270041   1.38778e-17  -0.856138
 0.83313    0.856138      2.77556e-17


exp(t ad(X))(Y):


3×3 Matrix{Float64}:
 0.0       -0.270041  -0.83313
 0.270041   0.0       -0.856138
 0.83313    0.856138   0.0


一致するか？ -> true
